# 融合模型：SBERT (標題) + MLP (結構化特徵) → 4 類分類

## 架構說明

```
標題 (text)
    └─► SBERT encoder ──► title_embedding (384維)
                                              \
                                               ► Fusion MLP ──► 4 classes
                                              /
結構化特徵 (24維)
    └─► MLP backbone ──► feature_embedding (64維)
```

**輸出 4 個類別：**
- Class 0: < 10,000
- Class 1: 10,000 – 99,999
- Class 2: 100,000 – 299,999
- Class 3: ≥ 300,000

## 1. 安裝相依套件

In [84]:
!pip install torch sentence-transformers numpy isodate -q

## 2. Import

In [85]:
import torch
import torch.nn as nn
import numpy as np
from sentence_transformers import SentenceTransformer
from collections import OrderedDict

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

PyTorch version: 2.11.0+cu128
Using device: cuda


In [86]:
!git clone https://github.com/yitingCurry/DL_Data.git

fatal: destination path 'DL_Data' already exists and is not an empty directory.


## 3. 載入原始模型權重

In [87]:
# ── 請修改成你的 .pt 檔路徑 ──
SBERT_MODEL_PATH = "DL_Data/title_SBERT_classification.pt"
MLP_MODEL_PATH   = "DL_Data/mlp_cls_model.pt"

# 載入 SBERT 模型資訊
sbert_ckpt = torch.load(SBERT_MODEL_PATH, map_location='cpu', weights_only=False)
print("SBERT checkpoint keys:", list(sbert_ckpt.keys()))
print(f"  SBERT model name : {sbert_ckpt['sbert_model']}")
print(f"  embedding_dim    : {sbert_ckpt['embedding_dim']}")
print(f"  num_classes      : {sbert_ckpt['num_classes']}")
print(f"  class_labels     : {sbert_ckpt['class_labels']}")

# 載入 MLP state dict
mlp_state_dict = torch.load(MLP_MODEL_PATH, map_location='cpu', weights_only=False)
MLP_INPUT_DIM  = mlp_state_dict['mlp.0.weight'].shape[1]   # 24
print(f"\nMLP input_dim: {MLP_INPUT_DIM}")
print("MLP layer shapes:")
for k, v in mlp_state_dict.items():
    if 'weight' in k and 'running' not in k and 'num_batches' not in k:
        print(f"  {k}: {v.shape}")

SBERT checkpoint keys: ['version', 'task', 'sbert_model', 'embedding_dim', 'head_state_dict', 'feature_mean', 'feature_std', 'metrics', 'num_classes', 'bin_mode', 'bin_edges', 'class_labels']
  SBERT model name : paraphrase-multilingual-MiniLM-L12-v2
  embedding_dim    : 384
  num_classes      : 4
  class_labels     : {'0': '<10,000', '1': '10,000-99,999', '2': '100,000-299,999', '3': '>=300,000'}

MLP input_dim: 24
MLP layer shapes:
  mlp.0.weight: torch.Size([1024, 24])
  mlp.1.weight: torch.Size([1024])
  mlp.4.weight: torch.Size([512, 1024])
  mlp.5.weight: torch.Size([512])
  mlp.8.weight: torch.Size([256, 512])
  mlp.9.weight: torch.Size([256])
  mlp.12.weight: torch.Size([128, 256])
  mlp.13.weight: torch.Size([128])
  mlp.16.weight: torch.Size([64, 128])
  mlp.18.weight: torch.Size([4, 64])


## 4. 定義各子網路

In [88]:
# ── MLP Backbone（原始架構，24維 → 64維中間表示）──
class MLPBackbone(nn.Module):
    """
    與原始 mlp_cls_model.pt 相同的網路結構，
    但只取到倒數第二層（64 維輸出），不含最終分類頭。

    原始架構：
      Linear(24→1024) → BN → ReLU → Dropout
      Linear(1024→512) → BN → ReLU → Dropout
      Linear(512→256)  → BN → ReLU → Dropout
      Linear(256→128)  → BN → ReLU → Dropout
      Linear(128→64)   → ReLU          ← backbone 輸出
      Linear(64→4)                     ← 原分類頭（融合後不用）
    """
    def __init__(self, input_dim: int = 24, dropout: float = 0.3):
        super().__init__()
        self.mlp = nn.Sequential(
            # Block 1
            nn.Linear(input_dim, 1024),   # mlp.0
            nn.BatchNorm1d(1024),          # mlp.1
            nn.ReLU(),                     # mlp.2
            nn.Dropout(dropout),           # mlp.3
            # Block 2
            nn.Linear(1024, 512),          # mlp.4
            nn.BatchNorm1d(512),           # mlp.5
            nn.ReLU(),                     # mlp.6
            nn.Dropout(dropout),           # mlp.7
            # Block 3
            nn.Linear(512, 256),           # mlp.8
            nn.BatchNorm1d(256),           # mlp.9
            nn.ReLU(),                     # mlp.10
            nn.Dropout(dropout),           # mlp.11
            # Block 4
            nn.Linear(256, 128),           # mlp.12
            nn.BatchNorm1d(128),           # mlp.13
            nn.ReLU(),                     # mlp.14
            nn.Dropout(dropout),           # mlp.15
            # Block 5 (backbone output)
            nn.Linear(128, 64),            # mlp.16
            nn.ReLU(),                     # mlp.17
            # mlp.18 = Linear(64→4) 留給融合頭，不放在這裡
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mlp(x)   # (B, 64)


# ── Fusion Model ──
class FusedModel(nn.Module):
    """
    輸入：
      - title_embedding : (B, 384)  — 由 SBERT 編碼的標題向量
      - struct_features : (B, 24)   — 結構化特徵（訂閱數、影片數等）

    架構：
      title_embedding  (384) ──┐
                                ├─ concat (448) → Linear(448→256) → BN → ReLU
      MLP backbone     (64)  ──┘               → Linear(256→128) → BN → ReLU
                                               → Linear(128→4)   → logits
    """
    def __init__(
        self,
        sbert_dim:    int   = 384,
        mlp_backbone_dim: int = 64,
        num_classes:  int   = 4,
        mlp_input_dim: int  = 24,
        dropout: float      = 0.3,
    ):
        super().__init__()
        self.mlp_backbone = MLPBackbone(input_dim=mlp_input_dim, dropout=dropout)
        fused_dim = sbert_dim + mlp_backbone_dim   # 384 + 64 = 448

        self.fusion_head = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(
        self,
        title_embedding: torch.Tensor,   # (B, 384)
        struct_features: torch.Tensor,   # (B, 24)
    ) -> torch.Tensor:
        feat = self.mlp_backbone(struct_features)          # (B, 64)
        fused = torch.cat([title_embedding, feat], dim=1)  # (B, 448)
        logits = self.fusion_head(fused)                   # (B, 4)
        return logits

print("✅ 網路定義完成")

✅ 網路定義完成


## 5. 載入預訓練權重到融合模型

In [89]:
def load_fused_model(sbert_ckpt_path: str, mlp_ckpt_path: str, device) -> FusedModel:
    sbert_ckpt  = torch.load(sbert_ckpt_path, map_location='cpu', weights_only=False)
    mlp_sd      = torch.load(mlp_ckpt_path,   map_location='cpu', weights_only=False)

    mlp_input_dim = mlp_sd['mlp.0.weight'].shape[1]   # 24
    sbert_dim     = sbert_ckpt['embedding_dim']        # 384
    num_classes   = sbert_ckpt['num_classes']          # 4

    model = FusedModel(
        sbert_dim=sbert_dim,
        mlp_backbone_dim=64,
        num_classes=num_classes,
        mlp_input_dim=mlp_input_dim,
    )

    # ── 載入 MLP backbone 權重（排除最後分類頭 mlp.18）──
    backbone_sd = OrderedDict()
    for k, v in mlp_sd.items():
        if not k.startswith('mlp.18'):   # 18 是原始分類頭，fusion 後不需要
            backbone_sd[k] = v

    missing, unexpected = model.mlp_backbone.load_state_dict(backbone_sd, strict=False)
    print(f"MLP backbone 載入完成")
    if missing:     print(f"  Missing keys : {missing}")
    if unexpected:  print(f"  Unexpected   : {unexpected}")

    # ── fusion_head 為隨機初始化（需要 fine-tune）──
    print("Fusion head: 隨機初始化（需在新資料上 fine-tune）")

    model.to(device)
    return model, sbert_ckpt


fused_model, sbert_ckpt = load_fused_model(SBERT_MODEL_PATH, MLP_MODEL_PATH, device)
print(f"\n模型參數量: {sum(p.numel() for p in fused_model.parameters()):,}")
print(fused_model)

MLP backbone 載入完成
Fusion head: 隨機初始化（需在新資料上 fine-tune）

模型參數量: 875,844
FusedModel(
  (mlp_backbone): MLPBackbone(
    (mlp): Sequential(
      (0): Linear(in_features=24, out_features=1024, bias=True)
      (1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
      (4): Linear(in_features=1024, out_features=512, bias=True)
      (5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): Dropout(p=0.3, inplace=False)
      (8): Linear(in_features=512, out_features=256, bias=True)
      (9): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (10): ReLU()
      (11): Dropout(p=0.3, inplace=False)
      (12): Linear(in_features=256, out_features=128, bias=True)
      (13): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (14): ReLU()
      (15): Dropout(p=0.3, inplace=Fa

## 6. 載入 SBERT encoder

In [90]:
sbert_encoder = SentenceTransformer(sbert_ckpt['sbert_model'], device=str(device))

# 載入原始模型儲存的 feature 正規化參數
feature_mean = torch.tensor(sbert_ckpt['feature_mean'], dtype=torch.float32)
feature_std  = torch.tensor(sbert_ckpt['feature_std'],  dtype=torch.float32)

CLASS_LABELS = sbert_ckpt['class_labels']
print(f"SBERT encoder 載入完成: {sbert_ckpt['sbert_model']}")
print(f"class_labels: {CLASS_LABELS}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

SBERT encoder 載入完成: paraphrase-multilingual-MiniLM-L12-v2
class_labels: {'0': '<10,000', '1': '10,000-99,999', '2': '100,000-299,999', '3': '>=300,000'}


## 7. 推論函式

## 7.1. 資料預處理函數

In [91]:
import re
import pandas as pd
import isodate

def parse_iso8601_duration(duration_str):
    """Parses ISO 8601 duration string and returns total seconds."""
    if not isinstance(duration_str, str):
        return 0
    try:
        duration = isodate.parse_duration(duration_str)
        return duration.total_seconds()
    except Exception:
        return 0

def parse_published_at(published_at_str):
    """Parses published_at string and returns hour and weekday."""
    try:
        dt = pd.to_datetime(published_at_str)
        return {'pub_hour': dt.hour, 'pub_weekday': dt.dayofweek}
    except Exception:
        return {'pub_hour': 0, 'pub_weekday': 0}

## 7.2. 載入並預處理資料

In [92]:
DATA_PATH = 'DL_Data/tw_youtube_videos.jsonl'

def preprocess_data(data_path):
    data = []
    with open(data_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    df = pd.DataFrame(data)

    # 1. 移除 'category_id' == 10 的資料
    if 'category_id' in df.columns:
        df['category_id'] = pd.to_numeric(df['category_id'], errors='coerce')
        df = df[df['category_id'] != 10].copy()

    # 2. 處理類別型特徵：One-hot Encoding for category_id
    df['category_id'] = df['category_id'].astype(str)
    df_encoded = pd.get_dummies(df, columns=['category_id'], prefix='cat')

    # 3. 數值特徵轉換
    if 'duration_iso8601' in df.columns:
        df_encoded['video_duration_sec'] = df['duration_iso8601'].apply(parse_iso8601_duration)

    if 'published_at' in df.columns:
        pub_info = df['published_at'].apply(lambda x: pd.Series(parse_published_at(str(x))))
        df_encoded = pd.concat([df_encoded, pub_info], axis=1)

        # 週期性編碼 (Cyclic Encoding) for pub_hour and pub_weekday
        df_encoded['pub_hour_sin'] = np.sin(2 * np.pi * df_encoded['pub_hour'] / 24.0)
        df_encoded['pub_hour_cos'] = np.cos(2 * np.pi * df_encoded['pub_hour'] / 24.0)
        df_encoded['pub_weekday_sin'] = np.sin(2 * np.pi * df_encoded['pub_weekday'] / 7.0)
        df_encoded['pub_weekday_cos'] = np.cos(2 * np.pi * df_encoded['pub_weekday'] / 7.0)

    # 確保數值型欄位為數值，並處理潛在的非數值資料
    numeric_cols_to_process = ['subscriber_count', 'description_length', 'channel_view_count', 'channel_video_count', 'tags_count', 'like_count', 'comment_count', 'favorite_count']
    for col in numeric_cols_to_process:
        if col in df_encoded.columns:
            df_encoded[col] = pd.to_numeric(df_encoded[col], errors='coerce').fillna(0)

    # subscriber_count / duration / description 壓縮至 log scale
    df_encoded['subscriber_count'] = np.log1p(df_encoded['subscriber_count'])
    df_encoded['video_duration_sec'] = np.log1p(df_encoded['video_duration_sec'])
    df_encoded['description_length'] = np.log1p(df_encoded['description_length'])
    df_encoded['channel_view_count'] = np.log1p(df_encoded['channel_view_count'])
    df_encoded['channel_video_count'] = np.log1p(df_encoded['channel_video_count'])

    # Define the mapping for class labels
    bin_edges = [10000, 100000, 300000]
    def map_view_count_to_class(view_count):
        if view_count < bin_edges[0]:
            return 0
        elif view_count < bin_edges[1]:
            return 1
        elif view_count < bin_edges[2]:
            return 2
        else:
            return 3

    df_encoded['view_count_class'] = df_encoded['view_count'].astype(int).apply(map_view_count_to_class)

    # 4. 建立特徵清單 (包含 One-hot 後的欄位)
    all_cat_cols = [col for col in df_encoded.columns if col.startswith('cat_')]
    # Ensure only category_id columns are included, excluding potential other 'cat_' prefixes from other sources if any
    CAT_COLS = [col for col in all_cat_cols if re.match(r'cat_\d+$', col)]

    NUM_COLS = [
        'subscriber_count',
        'video_duration_sec',
        'tags_count',
        'description_length',
        'pub_hour_sin',
        'pub_hour_cos',
        'pub_weekday_sin',
        'pub_weekday_cos',
        'channel_view_count',
        'channel_video_count',
    ]

    FEATURE_COLS = CAT_COLS + NUM_COLS

    # Ensure all FEATURE_COLS exist, fill missing with 0 and add if not present
    for col in FEATURE_COLS:
        if col not in df_encoded.columns:
            df_encoded[col] = 0.0 # Add missing columns with default value

    X = df_encoded[FEATURE_COLS].values.astype(np.float32)
    y = df_encoded['view_count_class'].values.astype(np.int64)
    titles = df_encoded['title'].tolist()

    return titles, X, y, FEATURE_COLS

# Load and preprocess data
import json
all_titles, all_features, all_labels, FEATURE_COLS = preprocess_data(DATA_PATH)
print(f"Total samples: {len(all_titles)}")
print(f"Number of features: {all_features.shape[1]}")
print(f"Feature columns: {FEATURE_COLS}")

# Update MLP_INPUT_DIM based on processed data
MLP_INPUT_DIM = all_features.shape[1]
print(f"Updated MLP_INPUT_DIM: {MLP_INPUT_DIM}")

Total samples: 10254
Number of features: 24
Feature columns: ['cat_1', 'cat_15', 'cat_17', 'cat_19', 'cat_2', 'cat_20', 'cat_22', 'cat_23', 'cat_24', 'cat_25', 'cat_26', 'cat_27', 'cat_28', 'cat_29', 'subscriber_count', 'video_duration_sec', 'tags_count', 'description_length', 'pub_hour_sin', 'pub_hour_cos', 'pub_weekday_sin', 'pub_weekday_cos', 'channel_view_count', 'channel_video_count']
Updated MLP_INPUT_DIM: 24


## 7.3. 數據分割與 SBERT 嵌入

In [93]:
from sklearn.model_selection import train_test_split

# Split data into training and validation sets
train_titles, val_titles, train_features, val_features, train_labels, val_labels = \
    train_test_split(all_titles, all_features, all_labels, test_size=0.2, random_state=42, stratify=all_labels)

print(f"Train samples: {len(train_titles)}, Val samples: {len(val_titles)}")

# Generate SBERT embeddings for train and validation titles
print("Generating SBERT embeddings for training titles...")
train_title_embeddings = sbert_encoder.encode(train_titles, convert_to_tensor=True, device=str(device)).cpu().numpy()
print("Generating SBERT embeddings for validation titles...")
val_title_embeddings = sbert_encoder.encode(val_titles, convert_to_tensor=True, device=str(device)).cpu().numpy()

print("SBERT embeddings generated.")

Train samples: 8203, Val samples: 2051
Generating SBERT embeddings for training titles...
Generating SBERT embeddings for validation titles...
SBERT embeddings generated.


In [94]:
def normalize_features(raw_features: np.ndarray) -> torch.Tensor:
    """
    使用訓練時儲存的 mean/std 對結構化特徵做標準化。
    raw_features: shape (N, 24) 的 numpy array
    """
    t = torch.tensor(raw_features, dtype=torch.float32)
    return (t - feature_mean) / (feature_std + 1e-8)


@torch.no_grad()
def predict(
    titles: list[str],
    struct_features: np.ndarray,   # shape: (N, 24)
    model: FusedModel,
    encoder: SentenceTransformer,
    normalize: bool = True,
) -> dict:
    """
    輸入：
      titles          : 標題字串列表，長度 N
      struct_features : 結構化特徵，shape (N, 24)
      normalize       : 是否套用 feature_mean/std 正規化

    輸出：dict 包含
      - pred_classes  : 預測的 class index (N,)
      - pred_labels   : 對應的 class label
      - probabilities : 各類別機率 (N, 4)
    """
    model.eval()

    # 1. SBERT 編碼標題
    title_emb = encoder.encode(titles, convert_to_tensor=True, device=str(device))  # (N, 384)
    title_emb = title_emb.float()

    # 2. 結構化特徵正規化
    if normalize:
        feats = normalize_features(struct_features).to(device)  # (N, 24)
    else:
        feats = torch.tensor(struct_features, dtype=torch.float32).to(device)

    # 3. 前向傳播
    logits = model(title_emb, feats)       # (N, 4)
    probs  = torch.softmax(logits, dim=1)  # (N, 4)
    preds  = torch.argmax(probs, dim=1)    # (N,)

    return {
        'pred_classes':  preds.cpu().numpy(),
        'pred_labels':   [CLASS_LABELS[str(p)] for p in preds.cpu().numpy()],
        'probabilities': probs.cpu().numpy(),
        'logits':        logits.cpu().numpy(),
    }

print("✅ 推論函式定義完成")

✅ 推論函式定義完成


## 9. Fine-tune Fusion Head（可選）

fusion head 是新增的層，需要在你的標記資料上 fine-tune。  
你可以選擇：
- **只訓練 fusion head**（MLP backbone & SBERT 凍結）← 建議先從這裡開始
- **全部解凍** 端對端 fine-tune

In [96]:
# ── 訓練設定 ──
FREEZE_BACKBONE = True    # True = 只訓練 fusion head
EPOCHS          = 10
LR              = 1e-3
BATCH_SIZE      = 32

# 凍結 backbone
if FREEZE_BACKBONE:
    for param in fused_model.mlp_backbone.parameters():
        param.requires_grad = False
    print("MLP backbone 已凍結，只訓練 fusion head")
else:
    for param in fused_model.parameters():
        param.requires_grad = True
    print("全部參數解凍，端對端 fine-tune")

trainable = sum(p.numel() for p in fused_model.parameters() if p.requires_grad)
print(f"可訓練參數量: {trainable:,}")

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, fused_model.parameters()),
    lr=LR, weight_decay=1e-4
)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

MLP backbone 已凍結，只訓練 fusion head
可訓練參數量: 149,124


In [100]:
from torch.utils.data import Dataset, DataLoader

class TitleFeatureDataset(Dataset):
    """
    Dataset 包含：
      title_embeddings : (N, 384) - 預先用 SBERT 編碼好的標題向量
      struct_features  : (N, 24)  - 標準化後的結構化特徵
      labels           : (N,)     - 整數 class label (0~3)
    """
    def __init__(self, title_embeddings, struct_features, labels):
        self.title_emb = torch.tensor(title_embeddings, dtype=torch.float32)
        self.feats     = torch.tensor(struct_features,  dtype=torch.float32)
        self.labels    = torch.tensor(labels,           dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.title_emb[idx], self.feats[idx], self.labels[idx]


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for title_emb, feats, labels in loader:
        title_emb, feats, labels = title_emb.to(device), feats.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(title_emb, feats)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for title_emb, feats, labels in loader:
        title_emb, feats, labels = title_emb.to(device), feats.to(device), labels.to(device)
        logits = model(title_emb, feats)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


# ──────────────────────────────────────────────
# 請換成你的真實資料：
#
# train_titles    : list of str
# train_features  : np.ndarray, shape (N_train, 24)，未標準化原始值
# train_labels    : np.ndarray, shape (N_train,)，整數 0~3
# val_titles, val_features, val_labels 同理
# ──────────────────────────────────────────────

# The data loaders (train_dl, val_dl) are now created in the next cell (fd1cd3ff)
# using the actual preprocessed data (train_title_embeddings, train_features, etc.).
# The dummy data section below is no longer needed.

print(f"Train size: {len(train_ds)}, Val size: {len(val_ds)}")


Train size: 9636, Val size: 2410


In [ ]:
train_ds = TitleFeatureDataset(train_title_embeddings, train_features, train_labels)
val_ds   = TitleFeatureDataset(val_title_embeddings,   val_features,   val_labels)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

print(f"Train size: {len(train_ds)}, Val size: {len(val_ds)}")

In [ ]:
# ── 訓練迴圈 ──
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(fused_model, train_dl, optimizer, criterion, device)
    va_loss, va_acc = eval_epoch(fused_model, val_dl, criterion, device)
    scheduler.step()

    flag = ""
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(fused_model.state_dict(), "fused_model_best.pt")
        flag = " ← best"

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train loss={tr_loss:.4f} acc={tr_acc:.2%} | "
          f"val loss={va_loss:.4f} acc={va_acc:.2%}{flag}")

print(f"\n最佳 Val Acc: {best_val_acc:.2%}")

## 10. 混淆矩陣 (Confusion Matrix)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# 獲取驗證集上的預測結果
val_preds_list = []
val_labels_list = []

fused_model.eval()
with torch.no_grad():
    for title_emb, feats, labels in val_dl:
        title_emb, feats, labels = title_emb.to(device), feats.to(device), labels.to(device)
        logits = fused_model(title_emb, feats)
        preds = torch.argmax(logits, dim=1)
        val_preds_list.extend(preds.cpu().numpy())
        val_labels_list.extend(labels.cpu().numpy())

# 計算混淆矩陣
cm = confusion_matrix(val_labels_list, val_preds_list)

# 繪製混淆矩陣
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[CLASS_LABELS[str(i)] for i in sorted(CLASS_LABELS.keys())],
            yticklabels=[CLASS_LABELS[str(i)] for i in sorted(CLASS_LABELS.keys())])
plt.xlabel('預測類別')
plt.ylabel('真實類別')
plt.title('混淆矩陣')
plt.show()

# 打印分類報告
print("分類報告：")
print(classification_report(val_labels_list, val_preds_list, target_names=[CLASS_LABELS[str(i)] for i in sorted(CLASS_LABELS.keys())]))

## 11. 儲存最終融合模型

## 12. 從儲存檔重新載入並推論

## 10. 儲存最終融合模型

In [ ]:
save_payload = {
    'version':        2,
    'task':           'classification',
    'sbert_model':    sbert_ckpt['sbert_model'],
    'embedding_dim':  sbert_ckpt['embedding_dim'],
    'mlp_input_dim':  24,
    'num_classes':    sbert_ckpt['num_classes'],
    'class_labels':   sbert_ckpt['class_labels'],
    'bin_edges':      sbert_ckpt['bin_edges'],
    'feature_mean':   sbert_ckpt['feature_mean'],
    'feature_std':    sbert_ckpt['feature_std'],
    'fused_model_state_dict': fused_model.state_dict(),
}

torch.save(save_payload, "fused_model_final.pt")
print("✅ 融合模型已儲存至 fused_model_final.pt")

## 11. 從儲存檔重新載入並推論

In [ ]:
def load_saved_fused_model(path: str, device):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    model = FusedModel(
        sbert_dim=ckpt['embedding_dim'],
        mlp_backbone_dim=64,
        num_classes=ckpt['num_classes'],
        mlp_input_dim=ckpt['mlp_input_dim'],
    )
    model.load_state_dict(ckpt['fused_model_state_dict'])
    model.to(device).eval()
    encoder = SentenceTransformer(ckpt['sbert_model'], device=str(device))
    return model, encoder, ckpt

# 載入並推論
loaded_model, loaded_encoder, loaded_ckpt = load_saved_fused_model("fused_model_final.pt", device)

res = predict(
    titles=["Top 10 AI tools you must know in 2025"],
    struct_features=np.random.randn(1, 24).astype(np.float32),
    model=loaded_model,
    encoder=loaded_encoder,
    normalize=True,
)
print(f"預測類別: {res['pred_labels'][0]}")
print(f"各類別機率: {dict(zip(CLASS_LABELS.values(), res['probabilities'][0]))}")